# Справка по статистическим функциям и методам
---

> **Описательная статистика** – это не просто набор чисел, а ключ к пониманию ваших данных. Качественный предварительный анализ поможет избежать многих проблем на этапе построения моделей ИИ и сэкономит вам много времени и ресурсов в будущем.

**Таблица**. *Сводная таблица статистических функций*
|Статистика| Метод `Pandas`	| Функция `SciPy`| Функция `NumPy`|	Примечания|
|:-:| :-: | :-:  | - |-|
|**Среднее**|`pd.series.mean()`|	`stats.tmean(data)`|	`np.mean(data)`||
|**Медиана**|`pd.series.median()`|`stats.scoreatpercentile(data, 50)`|	`np.median(data)`||
|**Мода**|`pd.series.mode()`|	`stats.mode(data).mode[0]`|нет встроенной функции|`SciPy` (осторожно!) возвращает только наименьшую моду|
|**Дисперсия**|`pd.series.var()`|`stats.tvar(data)`|	`np.var(data, ddof=1)`| В `Pandas` и `stats` по умолчанию несмещенная (`ddof=1`)|
|**Стд. отклонение**|`pd.series.std()`|`stats.tstd(data)`|	`np.std(data, ddof=1)`||
|**Моменты**|нет встроенной функции|`stats.moment(data, k)`|нет встроенной функции||
|**Асимметрия**|`pd.series.skew()`|	`stats.skew(data)`|	нет встроенной функции|`Pandas` всегда возвращает **несмещенную** (`unbiased`) оценку коэффициента асимметрии и не предоставляет параметра для изменения, `SciPy` дает выбор через параметр `bias`|
|**Эксцесс**|`pd.series.kurtosis()`|`stats.kurtosis(data)`|нет встроенной функции|	в `SciPy`: `Fisher=True` → эксцесс Фишера, `Fisher=False` → эксцесс Пирсона|
|**процентили (квантили)**|`pd.series.quantile(q)`|`stats.scoreatpercentile(data, p)`|`np.percentile(data, p)` `np.quantile(data, q)`|Квантили: $0–1$, процентили: $0–100$|


**О дисперсии и стандартном отклонении**
* Если ваши данные – вся генеральная совокупность: используйте `ddof=0` (смещенная оценка).
* Если ваши данные – выборка из популяции: используйте `ddof=1` (несмещенная оценка).

*Из теории*: Деление на $n-1$ вместо $n$ (так называемая поправка *Бесселя*) делает оценку дисперсии несмещенной для выборки. Деление на $n$ дает смещенную оценку, что подходит для описания самой выборки. Если вы хотите оценить изменчивость в генеральной совокупности по выборке, используйте `ddof=1` (деление на $n-1$).

**Об эксцессе**
В `SciPy` функция `kurtosis()` по умолчанию возвращает избыточный эксцесс, где для нормального распределения значение равно $0$. Это современный стандарт в статистике. Если вам нужен классический эксцесс Пирсона (где норма = $3$), используйте `fisher=False`.


**Функция**, рассчитывающая все основные статистики для ряда данных.
~~~python
import numpy as np
from scipy import stats
import pandas as pd

def calculate_all_stats(data, sample=True):
    """
    Рассчитывает все основные статистики для ряда данных.
    
    Параметры:
    data - список или массив чисел
    sample - True для выборки (ddof=1), False для совокупности (ddof=0)
    """
    ddof = 1 if sample else 0
    
    stats_dict = {
        'Среднее': np.mean(data),
        'Медиана': np.median(data),
        'Мода': pd.Series(data).mode().tolist(),
        'Дисперсия': np.var(data, ddof=ddof),
        'Стд. отклонение': np.std(data, ddof=ddof),
        'Асимметрия': stats.skew(data),
        'Эксцесс (избыточный)': stats.kurtosis(data, fisher=True),
        'Эксцесс (классический)': stats.kurtosis(data, fisher=False),
        'Минимум': np.min(data),
        'Максимум': np.max(data),
        'Размах': np.max(data) - np.min(data),
        'Квартиль 25%': np.percentile(data, 25),
        'Квартиль 75%': np.percentile(data, 75),
    }
    
    return stats_dict
~~~

## Расчет статистических характеристик

In [1]:
# Импорт необходимых библиотек
import numpy as np
import pandas as pd
import scipy.stats as stats

In [2]:
ls = [15, 18, 20, 22, 22, 25, 100]
data = np.array(ls)
ser = pd.Series(data)

### Среднее (`mean`)
___
* `numpy.mean(data)`  или `data.mean()`
* `pandas.series.mean()`
* `scipy.stats.tmean(data)`

In [ ]:
print("Математическое ожидание:")
print(f"    * numpy: {np.mean(data):.4f}")
print(f"    * pandas: {ser.mean():.4f}")
print(f"    * stats: {stats.tmean(data):.4f}")

Математическое ожидание:
    * numpy: 31.7143
    * pandas: 31.7143
    * stats: 31.7143


### Медиана (`median`)
---
* `numpy.median(data)`
* `pandas.series.median()`
* `stats.scoreatpercentile(data, 50)`

In [7]:
print("Медиана:")
print(f"    * numpy: {np.median(data):.4f}")
print(f"    * pandas: {ser.median():.4f}")
print(f"    * stats: {stats.scoreatpercentile(data, 50):.4f}")

Медиана:
    * numpy: 22.0000
    * pandas: 22.0000
    * stats: 22.0000


### Мода (`mode`)
---

* `pandas.series.mode()`
* `scipy.stats.mode()`

#### <u>`scipy.stats.mode()`</u>
> `scipy.stats.mode(data, keepdims=False)` – функция для нахождения моды – самого частого значения в наборе данных. Функция возвращает не просто число, а специальный объект `ModeResult`, который содержит два элемента:
>   * `mode` – само модальное значение;
>   * `count` – сколько раз это значение встретилось.

**Параметры функции**, которые делают ее гибкой для разных задач:
* `axis`  – отвечает за работу с многомерными данными. Представьте, что у вас не просто список, а таблица (матрица) чисел. С помощью этого параметра вы можете искать моду по строкам (`axis=1`), по столбцам (`axis=0`) или по всему массиву сразу (`axis=None`).
* `nan_policy` – правила обработки пропусков, определяет, что делать, если в данных встретилось `NaN` (не число):
    * `propagate` (по умолчанию) – если в данных есть `NaN`, то и мода будет `NaN`;
    * `raise` – программа остановится и выдаст ошибку;
    * `omit`– функция просто проигнорирует все `NaN` и посчитает моду по оставшимся числам.
* `keepdims` – сохранение размерности. Если `True`, то результат будет той же размерности, что и исходные данные (полезно для сложных вычислений).

**Ограничение, если мод несколько**

Если распределение мультимодальное, `scipy.stats.mode()` возвращиет **только одно, наименьшее значение**. Функция не сообщает, что есть и другая мода, поэтому всегда важно помнить об этой особенности при анализе данных.

#### <u>`pandas.series.mode()`</u>

> Метод `pandas.series.mode()` возвращает `DataFrame` с модальными значениями.  Если в данных несколько значений встречаются одинаково часто, `Pandas` вернет их все в отдельных строках – это принципиальное отличие от `SciPy`, где возвращается специальный объект `ModeResult`. Можно преобрзовать модальные значения в список с помощью метода `tolist()`. 

**Параметры** метода:
* `axis`  – определяет направление поиска моды: можно искать моду по строкам (`axis=1` или `axis='columns'`), по столбцам (`axis=0` или `axis='index'`). По умолчанию `axis=0` (поиск моды для каждого столбца).
* `numeric_only` – определяет, нодо ли включать в расчет только числовые столбцы (`numeric_only=True`) или включать нахождение моды и для других типов (`numeric_only=False`, стоит по умолчанию).
* `dropna` – определяет надо ли учитывать пропущенные значения `NaN` при поиске моды: `dropna=True` – игнорировать `NaN` (по умолчанию), `dropna=False`– учитывать `NaN`.

In [8]:
mode_data = stats.mode(data)
print(f"Мода: {mode_data.mode} (встречается {mode_data.count} раз)")

mode_ser = ser.mode()
print(f"Мода: {mode_ser}")

Мода: 22 (встречается 2 раз)
Мода: 0    22
dtype: int64


In [16]:
# Мультимодальный случай
temp = [3, 5, 5, 5, 5, 4, 4, 2, 3, 3, 4, 4, 4, 5]
tnp = np.array(temp)
tsr = pd.Series(temp)

mode_data = stats.mode(tnp)
print("Мода:")
print(f"    * stats: {mode_data.mode} (встречается {mode_data.count} раз)")
print(f"    * pandas: {tsr.mode().tolist()}")

print("Датафрейм со значениями моды")
display(tsr.mode())

Мода:
    * stats: 4 (встречается 5 раз)
    * pandas: [4, 5]
Датафрейм со значениями моды


0    4
1    5
dtype: int64

### Дисперсия 
---

* `numpy.var(data)`
* `pandas.series.var()`
* `stats.tvar(data)`

**Обратите внимание** в функции `numpy.var()` параметр `ddof` (*Delta Degrees of Freedom*) по умолчанию равен $0$, в результате будет получена смещенная оценка стандартного отклонения (деление на $n$, что подходит для описания самой выборки). При `ddof=1` мы получаем несмещенную оценку для выборки, что соответствует расчету в `pandas` (деление на $n-1$). 

In [12]:
print("Дисперсия (несмещенная, ddof=1): ")
print(f"    * numpy: {np.var(data, ddof=1):.2f}")   # ddof=1 для несмещенной оценки
print(f"    * pandas: {ser.var():.2f}")
print(f"    * stats: {stats.tvar(data, ddof=1):.2f}")

Дисперсия (несмещенная, ddof=1): 
    * numpy: 916.90
    * pandas: 916.90
    * stats: 916.90


In [13]:
print("Дисперсия (смещенная, ddof=0): ")
print(f"    * numpy: {np.var(data, ddof=0):.2f}")   # ddof=0 для смещенной оценки
print(f"    * pandas: {ser.var(ddof=0):.2f}")
print(f"    * stats: {stats.tvar(data, ddof=0):.2f}")

Дисперсия (смещенная, ddof=0): 
    * numpy: 785.92
    * pandas: 785.92
    * stats: 785.92


### Стандартное отклонение 
---

* `numpy.std(data)`
* `pandas.series.std()`
* `stats.tstd(data)`

**Обратите внимание** в функции `numpy.std()`параметр `ddof` (*Delta Degrees of Freedom*) по умолчанию равен $0$, в результате будет получена смещенная оценка стандартного отклонения (деление на $n$, что подходит для описания самой выборки). При `ddof=1` мы получаем несмещенную оценку для выборки, что соответствует расчету в `pandas` (деление на $n-1$). 

In [17]:
print("Стандартное отклоннение несмещенное, ddof=1:")
print(f"    * numpy: {np.std(data, ddof=1):.2f}")   # ddof=1 для несмещенной оценки
print(f"    * pandas: {ser.std():.2f}")
print(f"    * stats: {stats.tstd(data, ddof=1):.2f}")

Стандартное отклоннение несмещенное, ddof=1:
    * numpy: 30.28
    * pandas: 30.28
    * stats: 30.28


In [18]:
print("Стандартное отклоннение смещенное, ddof=0:")
print(f"    * numpy: {np.std(data, ddof=0):.2f}")   # ddof=1 для несмещенной оценки
print(f"    * pandas: {ser.std(ddof=0):.2f}")
print(f"    * stats: {stats.tstd(data, ddof=0):.2f}")

Стандартное отклоннение смещенное, ddof=0:
    * numpy: 28.03
    * pandas: 28.03
    * stats: 28.03


### Коэффициент вариации 
---

* `numpy` не имеет отдельной функции для коэффициента вариации, но его легко вычислить через `std()` и `mean()`: `np.std(data, ddof=1) / np.mean(data)) * 100`;
* в `pandas` принцип как в `numpy`, но функции работают с `Series/DataFrame`, и `std()` по умолчанию дает несмещенную оценку: `series.std() / series.mean() * 100`;
* в `scipy.stats` есть готовая функция `variation(data)`, ее параметры:
    * `data` – входные данные (массив, список);
    * `axis` – ось, вдоль которой считать (по умолчанию по всем элементам);
    * `ddof` – имеет тот же смысл (*Delta Degrees of Freedom*), что и в `numpy/pandas`: при `ddof=1` используется несмещенное стандартное отклонение (деление на $n-1$). По умолчанию `ddof=0`;
    * `nan_policy` – обработка пропусков (`propagate`, `raise`, `omit`).

**Таблица**. Интерпритация коэффициента вариации
|Значение коэффициента вариации| Интерпритация|
|:-:|-|
|$<10$%| Очень низкая изменчивость, данные однородны|
|$10-30$%|Умеренная изменчивость|
|$30-100$%|Высокая изменчивость|
|$>100$%|Очень высокая изменчивость, возможны выбросы|

**Важное предупреждение**. Коэффициент вариации $CV$ теряет смысл, если среднее значение близко к нулю (так как происходит деление на очень маленькое число, и $CV$ "взлетает" до бесконечности). Для данных с отрицательными значениями интерпретация $CV$ также может быть затруднена.

In [19]:
# Коэффициент вариации
print("Коэффициент вариации (несмещенный, ddof=1):")
print(f"    * numpy: {(np.std(data, ddof=1) / np.mean(data)) * 100:.2f}%")  
print(f"    * pandas: {ser.std() / ser.mean() * 100:.2f}%")
print(f"    * scipy.stats: {stats.variation(ser, ddof=1) * 100:.2f}%")

Коэффициент вариации (несмещенный, ddof=1):
    * numpy: 95.48%
    * pandas: 95.48%
    * scipy.stats: 95.48%


### Центральные моменты
---
* `stats.moment(data, k)`

In [20]:
moment_stats = stats.moment(data, moment=[2, 3, 4])
print("Моменты (через stats.moment):")
print(f"  2-й момент: {moment_stats[0]:.4f}")
print(f"  3-й момент: {moment_stats[1]:.4f}")
print(f"  4-й момент: {moment_stats[2]:.4f}")

Моменты (через stats.moment):
  2-й момент: 785.9184
  3-й момент: 43917.0962
  4-й момент: 3127870.3065


In [21]:
# Ручной расчет
mean = np.mean(data)

m2 = np.mean((data - mean)**2)  # второй центральный момент (он же смещенная дисперсия)
m3 = np.mean((data - mean)**3)  # третий центральный момент
m4 = np.mean((data - mean)**4)  # четвертый центральный момент

print("Центральные моменты (ручной расчет):")
print(f"   * второй (μ₂): {m2:.4f}")
print(f"   * третий (μ₃): {m3:.4f}")
print(f"   * четвертый (μ₄): {m4:.4f}")

Центральные моменты (ручной расчет):
   * второй (μ₂): 785.9184
   * третий (μ₃): 43917.0962
   * четвертый (μ₄): 3127870.3065


### Коэффициент асимметрии
---
* `pandas.series.skew()` → асимметрия (несмещенная)
* `stats.skew(data, bias=True)` → асимметрия (смещенная)
* `stats.skew(data, bias=False)` → асимметрия (несмещенная)
  
`Pandas` всегда возвращает **несмещенную** (`unbiased`) оценку (для выборки) коэффициента асимметрии, а `SciPy` по умолчанию тоже использует смещенную, но позволяет переключиться на несмещенную через параметр `bias=False`. Обе оценки статистически корректны, просто отвечают на разные вопросы.

* Смещенная (`bias=True`): *какова асимметрия именно этих чисел из выборки?*
* Несмещенная (`bias=False`): *какова асимметрия распределения, из которого взята эта выборка?*

Т.е. если набор рассматривается как все данные, то `bias=True`, если набор является выборкой из большего набора, то `bias=False`. 

|Используем смещенную оценку ( ***biased***)|Используем несмещенную оценку (***unbiased***)|
|:-|:-|
|1. Когда ваши данные – вся генеральная совокупность| 1.  Когда ваши данные – выборка из большей популяции|
|2. Когда вы просто описываете конкретную выборку без обобщения| 2. Когда вы хотите оценить истинную асимметрию распределения|
|3. Когда вы хотите, чтобы результаты совпадали с некоторыми статистическими пакетами|3. При малых выборках ($n < 30$) – для корректировки смещения |

In [22]:
print("Коэффициент асимметрии (несмещенная): ")
print(f"    * pandas: {ser.skew():.2f}")
print(f"    * stats: {stats.skew(data, bias=False):.2f}")

Коэффициент асимметрии (несмещенная): 
    * pandas: 2.58
    * stats: 2.58


In [23]:
print("Коэффициент асимметрии (смещенная): ")
print(f"    * stats: {stats.skew(data, bias=True):.2f}")

Коэффициент асимметрии (смещенная): 
    * stats: 1.99


In [24]:
skew_manual = m3 / (m2 ** 1.5)
print(f"Асимметрия (ручной расчет, смещенная): {skew_manual:.6f}")

# Поправка для несмещенной оценки асимметрии (для выборки)
n = len(data)
if n > 2:
    skew_unbiased = skew_manual * np.sqrt(n * (n - 1)) / (n - 2)
    print(f"Асимметрия (ручной расчет, несмещенная): {skew_unbiased:.6f}")

Асимметрия (ручной расчет, смещенная): 1.993276
Асимметрия (ручной расчет, несмещенная): 2.583581


### Эксцесс
---
* `pandas.series.kurtosis()` → избыточный эксцесс (`fisher=True`)
* `stats.kurtosis(data, fisher=True)` → избыточный эксцесс
* `stats kurtosis(data, fisher=False)` → классический эксцесс
  
В статистике существует **два основных определения эксцесса**:
* *Избыточный эксцесс* (*excess kurtosis*) – когда нормальное распределение имеет значение 0;
* *Классический эксцесс Пирсона* – когда нормальное распределение имеет значение 3.

`Pandas` `kurtosis`() и `SciPy` `kurtosis`() по умолчанию дают одинаковый результат – оба возвращают избыточный эксцесс (`fisher=True`), где нормальное распределение имеет значение $0$. Это современный стандарт в статистике.

Разница возникает только если в `SciPy` указать `fisher=False` – тогда получится классический эксцесс Пирсона (норма=$3$), который будет ровно на $3$ больше. Поэтому если вам нужен классический эксцесс Пирсона (где норма = $3$), используйте в `stats.kurtosis(data)` `fisher=False`. В `Pandas` `kurtosis`() не имеет такого параметра.

**Важно!**. **Эксцесс** – это диагностический инструмент, показывающий, насколько ваши данные "дружелюбны" к стандартным алгоритмам ИИ. ***Высокий эксцесс*** требует особого внимания к предобработке и выбору модели. Мониторинг эксцесса в процессе обучения помогает предотвратить нестабильность. Эксцесс можно использовать как признак в задачах, где важна форма распределения (анализ временных рядов, обнаружение аномалий).

**Таблица**. Интерпритация эксцесса
|Значение эксцесса| Интерпритация| Практические рекомендации для ИИ по значению эксцесса|
|:-:|-|-|
|$<0$|Данные "плоские", предсказуемые |Хорошо работают: линейная регрессия, логистическая регрессия, *SVM* с линейным ядром, нейросети с небольшим числом слоев. Есть риск недообучения для сложных моделей|
|$\approx0$|"Золотая середина"|Все модели и нейросети средней глубины работают хорошо; оптимально: градиентный бустинг (*XGBoost*, *LightGBM*). Можно использовать *StandardScaler*.")|
|$1-3$|Есть выбросы, но не критично|Хорошо работают: деревья решений и случайный лес, градиентный бустинг с регуляризацией, *SVM* с *rbf* ядром (после нормализации), нейросети с *Batch Normalization* и *Dropout*. *RobusScaler* может помочь, можно попробовать логарифмирование.|
|$3-10$|Сильные выбросы, тяжелые хвосты|Справятся: случайный лес (устойчив к выбросам), градиентный бустинг с ограничением глубины, изолирующий лес (для обнаружения аномалий), нейросети (*Huber loss* вместо *MSE*, *Adam* вместо *SGD*).  Избегайте: линейные модели без регуляризации. Высокий риск нестабильного обучения. Тяжелые хвосты – нужны степенные преобразования (*RobustScaler*) или логарифмирование для положительных данных|
|$>10$|Экстремальные выбросы||

In [25]:
print("Эксцесс (избыточный): ")
print(f"    * pandas: {ser.kurtosis():.2f}")
print(f"    * stats: {stats.kurtosis(data, fisher=True):.2f}")

Эксцесс (избыточный): 
    * pandas: 6.75
    * stats: 2.06


In [26]:
# Эксцесс (Kurtosis) - ручной расчет
# Классический эксцесс Пирсона (норма = 3)
kurt_pearson = m4 / (m2 ** 2)
# Избыточный эксцесс Фишера (норма = 0)
kurt_fisher = kurt_pearson - 3

print("Эксцесс (ручной расчет):")
print(f"   * классический (μ₄/μ₂²): {kurt_pearson:.6f}")
print(f"   * избыточный (μ₄/μ₂² - 3): {kurt_fisher:.6f}")

# Поправка для несмещенной оценки эксцесса (сложнее, формула для малых выборок)
n = len(data)
if n > 3:
    # Это приблизительная формула для несмещенного эксцесса
    kurt_unbiased = ((n - 1) * ((n + 1) * kurt_fisher + 6)) / ((n - 2) * (n - 3))
    print(f"   * правка на выборку: {kurt_unbiased:.6f}")

Эксцесс (ручной расчет):
   * классический (μ₄/μ₂²): 5.064002
   * избыточный (μ₄/μ₂² - 3): 2.064002
   * правка на выборку: 6.753605


### Процентили/ Квантили
---
* `pandas.series.quantile(q)`, $q \in [0, 1]$ 
* `stats.scoreatpercentile(data, p)`, $p \in [0, 100]$ 
* `numpy.percentile(data, p)`, $p \in [0, 100]$ 
* `numpy.quantile(data, q)`, $q \in [0, 1]$ 

In [27]:
print("75-й процентиль (0.75 квантиль):")
print(f"  * pandas: {ser.quantile(0.75)}")
print(f"  * stats: {stats.scoreatpercentile(data, 75)}")
print(f"  * numpy: {np.percentile(data, 75)}")
print(f"  * numpy: {np.quantile(data, 0.75)}")

75-й процентиль (0.75 квантиль):
  * pandas: 23.5
  * stats: 23.5
  * numpy: 23.5
  * numpy: 23.5


## Просмотр описательных статистик
---

### Описательные статистики в `SciPy`

In [28]:
from scipy import stats

data = [15, 18, 20, 22, 22, 25, 100]

desc = stats.describe(data)
print("stats.describe():")
print(f"  nobs: {desc.nobs}")
print(f"  minmax: {desc.minmax}")
print(f"  mean: {desc.mean:.4f}")
print(f"  variance: {desc.variance:.4f}")
print(f"  skewness: {desc.skewness:.6f}")
print(f"  kurtosis: {desc.kurtosis:.6f}")

stats.describe():
  nobs: 7
  minmax: (np.int64(15), np.int64(100))
  mean: 31.7143
  variance: 916.9048
  skewness: 1.993276
  kurtosis: 2.064002


### Описательные статистики в `Pandas`

In [29]:
import pandas as pd

data = [15, 18, 20, 22, 22, 25, 100]
df = pd.DataFrame(data, columns=['daily_sales'])

df.describe()

,daily_sales
count,7.000000
mean,31.714286
std,30.280435
min,15.000000
25%,19.000000
50%,22.000000
75%,23.500000
max,100.000000


### Собственная функция просмотра описательных статистик

In [30]:
import numpy as np
from scipy import stats
import pandas as pd

def calculate_all_stats(data, sample=True):
    """
    Рассчитывает все основные статистики для ряда данных.
    
    Параметры:
    data - список или массив чисел
    sample - True для выборки (ddof=1), False для совокупности (ddof=0)
    """
    ddof = 1 if sample else 0
    
    stats_dict = {
        'Среднее': np.mean(data),
        'Медиана': np.median(data),
        'Мода': pd.Series(data).mode().tolist(),
        'Дисперсия': np.var(data, ddof=ddof),
        'Стд. отклонение': np.std(data, ddof=ddof),
        'Асимметрия': stats.skew(data),
        'Эксцесс (избыточный)': stats.kurtosis(data, fisher=True),
        'Эксцесс (классический)': stats.kurtosis(data, fisher=False),
        'Минимум': np.min(data),
        'Максимум': np.max(data),
        'Размах': np.max(data) - np.min(data),
        'Квартиль 25%': np.percentile(data, 25),
        'Квартиль 75%': np.percentile(data, 75),
    }
    
    return stats_dict

data = [15, 18, 20, 22, 22, 25, 100]
results = calculate_all_stats(data)
for stat_name, value in results.items():
    if isinstance(value, list):
        print(f"{stat_name}: {value}")
    elif isinstance(value, float):
        print(f"{stat_name}: {value:.4f}")
    else:
        print(f"{stat_name}: {value}")

Среднее: 31.7143
Медиана: 22.0000
Мода: [22]
Дисперсия: 916.9048
Стд. отклонение: 30.2804
Асимметрия: 1.9933
Эксцесс (избыточный): 2.0640
Эксцесс (классический): 5.0640
Минимум: 15
Максимум: 100
Размах: 85
Квартиль 25%: 19.0000
Квартиль 75%: 23.5000
